In [931]:
from sage.all import gcd, PolynomialRing, factor
from fractions import Fraction
from functools import reduce

class WeightedProjectivePoint:
    def __init__(self, coordinates, weights, field):
        self.coordinates = tuple(coordinates)
        self.weights = weights
        self.field = field
        #self.normalize()

    def __repr__(self):
        return f"WeightedProjectivePoint {self.coordinates} over {self.field} with weights {self.weights}"

    def __str__(self):
        return str(self.coordinates)

    def __eq__(self, other):
        self.normalize()
        other.normalize()
        if self.coordinates == other.coordinates and self.weights == other.weights:
            return True
        else:
            return False

    def normalize(self):
        if self.field == QQ:
            self.coordinates = self.integer_representation()

        scale = self.weighted_gcd(self.coordinates, self.weights)
        ans = []
        for i in range(len(self.coordinates)):
            if self.field == QQ:
                ans.append(ZZ(self.coordinates[i] // (scale ** self.weights[i])))
            else:
                ans.append(self.field(self.coordinates[i] // (scale ** self.weights[i])))

        self.coordinates = ans



    def weighted_gcd(self, coordinates, weights):
            # ignore zeros
        coords = [abs(x) for x in coordinates if x != 0]
    
        if not coords:
            return 1
    
        g = gcd(coords)
    
        lam = 1
    
        for p, e in factor(g):
    
            bound = min(
                coordinates[i].valuation(p) // weights[i]
                for i in range(len(coordinates))
                if coordinates[i] != 0
            )
    
            lam *= p**bound
    
        return lam

    
    def integer_representation(self):
        #converts a weighted projective point with components in Q to Z

        # Convert to sage rational numbers
        ans = [QQ(x) for x in self.coordinates]
        
        # Clear denominators
        denoms = reduce(lcm, [x.denominator() for x in ans])
        for i in range(len(self.coordinates)):
            ans[i] = ZZ(ans[i] * (denoms ** self.weights[i]))
        
        return tuple(ans)
    
            

In [932]:
class WeightedProjectiveSpace:
    def __init__(self, field, weights):
        self.field = field
        self.weights = weights
        self.dimension = len(weights) - 1
        self.is_finite = field.is_finite()

    def __repr__(self):
        return f"WeightedProjectiveSpace over {self.field} with weights {self.weights}"

    def is_well_formed(self):
        return gcd(self.weights) == 1

    def make_well_formed(self):
        check = gcd(self.weights)
        if check == 1:
            return None
        else:
            for i in range(len(self.weights)):
                self.weights[i] = self.weights[i] // check

    def point(self, coordinates):
        return WeightedProjectivePoint(coordinates, self.weights, self.field)

    def rational_points(self):

        all_points = list(product(self.field, repeat=len(self.weights)))
        all_points.pop(0)
        points = []
        #runs through all possible points

        for point in all_points:
            x = WeightedProjectivePoint(point, self.weights, self.field)
            if x not in points:
                points.append(x)
        return points

        

In [933]:
class GradedPolynomialRing:
    def __init__(self, base_ring, weights):
        self.base_ring = base_ring
        self.weights = tuple(weights)
        self.n = len(weights)

        #n is the number of variables
        #so we get x0, x1, ... , xn-1
        self.ring = PolynomialRing(base_ring, self.n, 'x')
        self.vars = self.ring.gens()

    def __repr__(self):
        return f"Graded polynomial ring over {self.base_ring} with weights {self.weights}"

    def gens(self):
        return self.vars

    #returns the degree of a weighted monomial
    def degree_monomial(self, f):
        if len(f.exponents()) > 1:
            #throw error
            return None
            
        exp = f.exponents()[0]
        ans = 0
        for i in range(len(self.weights)):
            ans += self.weights[i] * exp[i]
        return ans

    #returns true if a given weighted polynomial is homogenous
    def is_homogenous(self, f):
        mons = f.monomials()
        deg = self.degree_monomial(mons[0])
        for mon in mons:
            if self.degree_monomial(mon) != deg:
                return False
        return True

    #returns the degree of a homogenous weighted polynomial
    def degree(self, f):
        if not self.is_homogenous(f):
            #throw error
            print("not homogenous")
            return None
        return self.degree_monomial(f.monomials()[0])

    #returns the degree d part of the weighted polynomial
    def homogenous_part(self, f, deg):
        ans = self.ring(0)
        mons = f.monomials()
        
        for mon in mons:
            print(mon)
            if self.degree_monomial(mon) == deg:
                ans += mon
        return ans